In [1]:
# import torch

# # !git clone https://github.com/facebookresearch/r3m.git
# !cd /kaggle/input/r3m/pytorch/default/1/r3m
# !pip install -e .


In [2]:
import torch
import torchvision
import r3m
work = '/kaggle/working/'
def get_resnet(name, weights=None, **kwargs):
    """
    name: resnet18, resnet34, resnet50
    weights: "IMAGENET1K_V1", "r3m"
    """
    # load r3m weights
    if (weights == "r3m") or (weights == "R3M"):
        return get_r3m(name=name, **kwargs)

    func = getattr(torchvision.models, name)
    resnet = func(weights=weights, **kwargs)
    resnet.fc = torch.nn.Identity()
    return resnet

def get_r3m(name, **kwargs):
    """
    name: resnet18, resnet34, resnet50
    """
    import r3m
    r3m.device = 'cpu'
    model = r3m.load_r3m(name)
    r3m_model = model.module
    resnet_model = r3m_model.convnet
    resnet_model = resnet_model.to('cpu')
    return resnet_model


In [3]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional

@dataclass
class SimpleConfig:
    """Simple configuration class with essential parameters"""
    vocab_size: int = 32000
    d_model: int = 768
    n_heads: int = 10
    n_layers: int = 12
    mlp_ratio: float = 4.0
    max_seq_length: int = 2048
    dropout: float = 0.1
    rope_theta: float = 10000.0
    
    @property
    def head_dim(self):
        return self.d_model // self.n_heads
    
    @property
    def hidden_size(self):
        return int(self.mlp_ratio * self.d_model)

class RMSLayerNorm(nn.Module):
    """RMS layer normalization"""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        
    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        return x * self.weight

class RotaryEmbedding(nn.Module):
    """Rotary positional embeddings"""
    def __init__(self, dim, max_seq_len=2048, theta=10000.0):
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.theta = theta
        
        cos, sin = self._compute_cos_sin_cache(max_seq_len)
        self.register_buffer("cos_cached", cos, persistent=False)
        self.register_buffer("sin_cached", sin, persistent=False)
        
    def _compute_cos_sin_cache(self, seq_len):
        inv_freq = 1.0 / (self.theta ** (torch.arange(0, self.dim, 2).float() / self.dim))
        seq = torch.arange(seq_len, dtype=torch.float)
        freqs = torch.outer(seq, inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        cos = emb.cos().view(1, 1, seq_len, self.dim)
        sin = emb.sin().view(1, 1, seq_len, self.dim)
        return cos, sin
        
    def forward(self, q, k, seq_len=None):
        if seq_len is None:
            seq_len = q.size(2)
        if seq_len > self.max_seq_len:
            cos, sin = self._compute_cos_sin_cache(seq_len)
            self.register_buffer("cos_cached", cos, persistent=False)
            self.register_buffer("sin_cached", sin, persistent=False)
        cos = self.cos_cached[:, :, :seq_len, :]
        sin = self.sin_cached[:, :, :seq_len, :]
        
        q1, q2 = q.chunk(2, dim=-1)
        k1, k2 = k.chunk(2, dim=-1)
        q_rot = torch.cat([-q2, q1], dim=-1)
        k_rot = torch.cat([-k2, k1], dim=-1)
        
        q = q * cos + q_rot * sin
        k = k * cos + k_rot * sin
        return q, k

class AttentionBlock(nn.Module):
    """Self-attention block similar to LLaMA"""
    def __init__(self, config: SimpleConfig):
        super().__init__()
        self.config = config
        
        self.norm = RMSLayerNorm(config.d_model)
        self.q_proj = nn.Linear(config.d_model, config.d_model, bias=False)
        self.k_proj = nn.Linear(config.d_model, config.d_model, bias=False)
        self.v_proj = nn.Linear(config.d_model, config.d_model, bias=False)
        self.o_proj = nn.Linear(config.d_model, config.d_model, bias=False)
        
        self.rope = RotaryEmbedding(
            config.head_dim, 
            max_seq_len=config.max_seq_length,
            theta=config.rope_theta
        )
        self.dropout = nn.Dropout(config.dropout)
        
    def forward(self, x):
        bsz, seqlen, _ = x.shape
        h = self.norm(x)
        q = self.q_proj(h).view(bsz, seqlen, self.config.n_heads, self.config.head_dim).transpose(1,2)
        k = self.k_proj(h).view(bsz, seqlen, self.config.n_heads, self.config.head_dim).transpose(1,2)
        v = self.v_proj(h).view(bsz, seqlen, self.config.n_heads, self.config.head_dim).transpose(1,2)
        
        q, k = self.rope(q, k, seqlen)
        scores = (q @ k.transpose(-2,-1)) / math.sqrt(self.config.head_dim)
        mask = torch.triu(torch.ones(seqlen, seqlen, device=x.device, dtype=torch.bool), diagonal=1)
        scores.masked_fill_(mask.unsqueeze(0).unsqueeze(0), -float('inf'))
        
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = attn @ v
        out = out.transpose(1,2).contiguous().view(bsz, seqlen, self.config.d_model)
        out = self.o_proj(out)
        return x + self.dropout(out)

class FeedForwardBlock(nn.Module):
    """Feed-forward block with SwiGLU activation"""
    def __init__(self, config: SimpleConfig):
        super().__init__()
        self.config = config
        
        self.norm = RMSLayerNorm(config.d_model)
        self.w1 = nn.Linear(config.d_model, config.hidden_size, bias=False)
        self.w2 = nn.Linear(config.d_model, config.hidden_size, bias=False)
        self.w3 = nn.Linear(config.hidden_size, config.d_model, bias=False)
        self.dropout = nn.Dropout(config.dropout)
        
    def forward(self, x):
        h = self.norm(x)
        h1 = self.w1(h)
        h2 = self.w2(h)
        hidden = F.silu(h1) * h2
        out = self.w3(hidden)
        return x + self.dropout(out)

class TransformerBlock(nn.Module):
    """Combined attention+FFN, with optional FiLM conditioning"""
    def __init__(
        self,
        config: SimpleConfig,
        cond_dim: Optional[int] = None,
        cond_predict_scale: bool = False
    ):
        super().__init__()
        self.config = config
        self.attention = AttentionBlock(config)
        self.feed_forward = FeedForwardBlock(config)

        # if cond_dim is given, build a little FiLM MLP
        self.cond_dim = cond_dim
        self.cond_predict_scale = cond_predict_scale
        if cond_dim is not None:
            out_ch = config.d_model * (2 if cond_predict_scale else 1)
            self.cond_encoder = nn.Sequential(
                nn.Mish(),
                nn.Linear(cond_dim, out_ch)
            )
        else:
            self.cond_encoder = None

    def forward(self, x, cond: Optional[torch.Tensor] = None):
        # --- self-attention + residual ---
        x = self.attention(x)

        # --- FiLM conditioning (if requested) ---
        if self.cond_encoder is not None:
            # cond: [batch, cond_dim]
            cond = cond.to(x.dtype)
            emb = self.cond_encoder(cond)  # [batch, out_ch]
            bsz = emb.size(0)
            if self.cond_predict_scale:
                # split to scale & bias
                emb = emb.view(bsz, 2, self.config.d_model)
                scale = emb[:,0].unsqueeze(1)   # [batch,1,d_model]
                bias  = emb[:,1].unsqueeze(1)
                x = scale * x + bias
            else:
                bias = emb.unsqueeze(1)        # [batch,1,d_model]
                x = x + bias

        # --- feed-forward + residual ---
        x = self.feed_forward(x)
        return x

class SimpleLLaDAModel(nn.Module):
    """Simplified LLaDA with FiLM conditioning"""
    def __init__(
        self,
        config: SimpleConfig,
        cond_dim: int,
        cond_predict_scale: bool = True
    ):
        super().__init__()
        self.config = config
        self.cond_dim = cond_dim
        self.cond_predict_scale = cond_predict_scale

        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(config, cond_dim, cond_predict_scale)
            for _ in range(config.n_layers)
        ])
        self.norm = RMSLayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids: torch.Tensor, cond: torch.Tensor):
        """
        input_ids: [batch, seq_len]
        cond:      [batch, cond_dim]
        """
        x = self.token_embedding(input_ids)   # [batch, seq_len, d_model]
        for block in self.blocks:
            x = block(x, cond)
        x = self.norm(x)
        logits = self.lm_head(x)
        return logits


def create_model(vocab_size=32000, d_model=100, n_heads=10, n_layers=12):
    """Helper function to create a model with default parameters"""
    config = SimpleConfig(
        vocab_size=vocab_size,
        d_model=d_model,
        n_heads=n_heads,
        n_layers=n_layers,
    )
    return SimpleLLaDAModel(config, 
                            cond_dim=529)
# Custom model



# vocab_size = 1024
# batch_size = 1
# seq_len = 29*16

# # Create random token indices (0 to vocab_size-1)
# input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
# cond = torch.rand((batch_size, 100))

# # Initialize model
# model = create_model(vocab_size=vocab_size)  # Uses default config

# # Forward pass
# with torch.no_grad():
#     logits = model(input_ids, cond)
    
# print("Output shape:", logits.shape)  # Should be (1, 5, 32000)
# print("Sample output:", logits[0, -1, :5])  # First 5 logits of last token



In [4]:
import os
import json
import numpy as np
import torch
from scipy.fft import dct, idct
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

class TemporalBPEProcessor:
    def __init__(
        self,
        scale: float = 100.0,
        min_token: int = 0,
        time_horizon: int | None = None,
        state_dim: int | None = None,
        normalization: str = "zscore",  # 'zscore' or 'minmax'
        mean: np.ndarray | None = None,
        std: np.ndarray | None = None,
        min_val: np.ndarray | None = None,
        max_val: np.ndarray | None = None,
    ):
        self.scale = scale
        self.min_token = min_token
        self.time_horizon = time_horizon
        self.state_dim = state_dim
        self.normalization = normalization
        self.mean = mean
        self.std = std
        self.min_val = min_val
        self.max_val = max_val
        self.called_time_horizon = time_horizon
        self.called_state_dim = state_dim
        self.T = 16
        self.mask_token_id = 0
        self.mnmx = 10

    def normalize(self, x: np.ndarray) -> np.ndarray:
        if self.normalization == "zscore":
            return (x - self.mean) / self.std
        elif self.normalization == "minmax":
            mnmx = self.max_val - self.min_val
            mnmx[mnmx<=1e-2]= 1
            return (x - self.min_val) / mnmx
        return x

    def _denormalize(self, x: np.ndarray) -> np.ndarray:
        if self.normalization == "zscore":
            return x * self.std + self.mean
        elif self.normalization == "minmax":
            mnmx = self.max_val - self.min_val
            mnmx[mnmx<=1e-2]= 1
            return x * (mnmx) + self.min_val
        return x

    def __call__(
        self,
        state_seq: np.ndarray,
        padding: bool = False,
        truncation: bool = False,
        max_length: int | None = None,
        return_tensors: str | None = None,
    ):
        """
        Tokenize a batch of state sequences via DCT quantization.

        Args:
            state_seq: np.ndarray of shape [B, T, D] or [T, D]
            padding: pad option (currently not used)
            truncation: truncate option (currently not used)
            max_length: length to pad or truncate to if return_tensors='pt'
            return_tensors: 'pt' for PyTorch tensors
        """
        if state_seq.ndim == 2:
            state_seq = state_seq[None, ...]

        batch_size, T, D = state_seq.shape
        self.called_time_horizon = T
        self.called_state_dim = D

        norm_seq = self.normalize(state_seq)
        #print(norm_seq.shape)
        coeff = dct(norm_seq, axis=1, norm='ortho')
        q = np.around(coeff * self.scale).astype(int)

        tokens: list[list[int]] = []
        for b in range(batch_size):
            flat = (q[b].flatten() - self.min_token).clip(min=0)
            tokens.append(flat.tolist())

        if return_tensors == 'pt':
            if max_length is None:
                max_length = max(len(t) for t in tokens)
            input_ids = torch.tensor(
                [t[:max_length] + [0] * max(0, max_length - len(t)) for t in tokens],
                dtype=torch.long
            )
            attention_mask = torch.tensor(
                [[1] * min(len(t), max_length) + [0] * max(0, max_length - len(t)) for t in tokens],
                dtype=torch.long
            )
            return {'input_ids': input_ids, 'attention_mask': attention_mask}
        if torch.tensor(tokens).max()>= 500:
           print(torch.tensor(tokens).max())

        return torch.tensor(tokens)+1

    def decode(
        self,
        token_ids: list[list[int]],
        state_dim: int | None = None,
    ) -> np.ndarray:
        D = state_dim or self.called_state_dim
        token_ids = (token_ids-1).clip(min = 0)
        decoded_seqs = []
        for ids in token_ids:
            arr = np.array(ids) + self.min_token
            arr = arr.reshape(-1, D)
            coeff = arr.astype(float) / self.scale
            rec = idct(coeff, axis=0, norm='ortho')
            rec = self._denormalize(rec)
            decoded_seqs.append(rec)
        return np.stack(decoded_seqs)
        
    @classmethod
    def load(cls, save_dir: str) -> "TemporalBPEProcessor":
        stats = np.load(os.path.join(save_dir, "normalization_stats.npz"))
        return cls(
            scale=float(stats["scale"]),
            min_token=int(stats["min_token"]),
            time_horizon=int(stats["time_horizon"]),
            state_dim=int(stats["state_dim"]),
            normalization=str(stats["normalization"]),
            mean=stats["mean"],
            std=stats["std"],
            min_val=stats["min_val"],
            max_val=stats["max_val"],
        )


In [5]:
import os
import json
import h5py
import numpy as np
import cv2
from torch.utils.data import Dataset, DataLoader
import torch

np.set_printoptions(precision=3, suppress=True)

import os
import json
import numpy as np
import torch
from torch.utils.data import Dataset

class ManiSkillSequenceDataset(Dataset):
    """
    Loads the prebuilt .npz shards for inputs and fetches the next pose+gripper horizon chunk
    Returns:
      {
        'obs': {'image': Tensor[C,H,W], 'pose': Tensor[16], 'gripper': Tensor[1]},
        'target_state': ndarray[state_horizon, D]  # future [pose, gripper] vectors
      }
    """
    def __init__(self, data_dir: str, transform=None, state_horizon: int = 16):
        self.data_dir = data_dir
        self.transform = transform
        self.state_horizon = state_horizon

        # load metadata
        meta_path = os.path.join(data_dir, 'meta.json')
        with open(meta_path, 'r') as f:
            meta = json.load(f)

        # episode info
        self.episodes = meta['episodes']
        lengths = [ep['length'] for ep in self.episodes]
        self.cumlen = np.cumsum([0] + lengths)

    def __len__(self):
        return int(self.cumlen[-1])

    def __getitem__(self, idx: int):
        # map global idx -> (episode, step)
        ep = int(np.searchsorted(self.cumlen, idx, side='right') - 1)
        step = idx - self.cumlen[ep]

        # load input arrays
        arr = np.load(os.path.join(self.data_dir, self.episodes[ep]['file']))
        img = arr['img'][step]
        pose = arr['pose'][step]
        grip = arr['grip'][step]

        # apply transform or default to [C,H,W] float
        if self.transform:
            img = self.transform(img)
        else:
            img = torch.from_numpy(img).permute(2, 0, 1).float().div(255.)

        # prepare tensors for obs
        pose = torch.from_numpy(pose).float()
        grip = torch.tensor([grip], dtype=torch.float32)

        # build future pose+gripper sequence
        poses = arr['pose']
        grips = arr['grip']
        horizon = []
        zero_pose = np.zeros_like(poses[0])
        zero_grip = np.zeros_like(np.array([grips[0]]))
        for i in range(self.state_horizon):
            future_idx = step + 1 + i
            if future_idx < len(poses):
                p = poses[future_idx]
                g = np.array([grips[future_idx]])
            else:
                p = zero_pose
                g = zero_grip
            horizon.append(np.concatenate([p, g], axis=0))

        target_state = np.stack(horizon, axis=0).astype(np.float32)

        return {
            'obs': {'image': img, 'pose': pose, 'gripper': grip},
            'target_state': target_state
        }


In [6]:
import argparse
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
# from model import create_model
# from data import ManiSkillSequenceDataset
# from tokeniser import TemporalBPEProcessor
# from vision_model_getter import get_resnet
from typing import Tuple, Sequence, Dict, Union, Optional, Callable
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler
from tqdm.auto import tqdm

# -----------------------------------------------------------------------------
# Diffusion-based language generation training script
# Non-autoregressive: fixed-length sequences, noisy (masked) inputs,
# model learns to denoise. Also includes an inference sampler.
# -----------------------------------------------------------------------------

def replace_submodules(
        root_module: nn.Module,
        predicate: Callable[[nn.Module], bool],
        func: Callable[[nn.Module], nn.Module]) -> nn.Module:
    """
    Replace all submodules selected by the predicate with
    the output of func.

    predicate: Return true if the module is to be replaced.
    func: Return new module to use.
    """
    if predicate(root_module):
        return func(root_module)

    bn_list = [k.split('.') for k, m
        in root_module.named_modules(remove_duplicate=True)
        if predicate(m)]
    for *parent, k in bn_list:
        parent_module = root_module
        if len(parent) > 0:
            parent_module = root_module.get_submodule('.'.join(parent))
        if isinstance(parent_module, nn.Sequential):
            src_module = parent_module[int(k)]
        else:
            src_module = getattr(parent_module, k)
        tgt_module = func(src_module)
        if isinstance(parent_module, nn.Sequential):
            parent_module[int(k)] = tgt_module
        else:
            setattr(parent_module, k, tgt_module)
    # verify that all modules are replaced
    bn_list = [k.split('.') for k, m
        in root_module.named_modules(remove_duplicate=True)
        if predicate(m)]
    assert len(bn_list) == 0
    return root_module

def replace_bn_with_gn(
    root_module: nn.Module,
    features_per_group: int=16) -> nn.Module:
    """
    Relace all BatchNorm layers with GroupNorm.
    """
    replace_submodules(
        root_module=root_module,
        predicate=lambda x: isinstance(x, nn.BatchNorm2d),
        func=lambda x: nn.GroupNorm(
            num_groups=x.num_features//features_per_group,
            num_channels=x.num_features)
    )
    return root_module

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import os

class DiffusionTrainer2:
    def __init__(self,
                 tokeniser,
                 seq_len: int = 128,
                 schedule_steps: int = 1000,
                 vocab_size=1000,
                 d_model=768,
                 n_heads=12,
                 n_layers=12,
                 device: str = 'cuda',
                 load_path: str = None):  # <-- Added load_path
        self.tokenizer = tokeniser
        self.seq_len = seq_len
        self.T = schedule_steps
        self.device = device
        self.vocab_size = vocab_size

        self.model = nn.ModuleDict({
            'vision_encoder': replace_bn_with_gn(get_resnet('resnet18')),
            'lldm': create_model(vocab_size=vocab_size, d_model=d_model, n_heads=n_heads, n_layers=n_layers)
        })

        if load_path and os.path.isfile(load_path):  # <-- Load model if specified
            print(f"Loading model from {load_path}")
            self.model.load_state_dict(torch.load(load_path, map_location=self.device))

        self.mask_id = self.tokenizer.mask_token_id
        self.optimizer = optim.AdamW(self.model.parameters(), lr=1e-4, weight_decay=1e-6)
        
    def linear_noise_schedule(self, t: torch.Tensor) -> torch.Tensor:
        """
        Compute mask probability p_mask(t) = (t+1)/T
        """
        return (t.float() + 1) / self.T
    def q_sample(self, x_start: torch.Tensor, t: torch.Tensor):
        """
        Forward diffusion: mask tokens with probability p_mask(t)
        """
        #print(t)
        p_mask = (t.float() + 1) / self.T#self.linear_noise_schedule(t)#.unsqueeze(-1)
        rand = torch.rand_like(x_start.float(), device=self.device)
        #print(rand.shape, p_mask.shape)
        mask_indices = rand < p_mask
        x_noisy = x_start.clone()
        x_noisy[mask_indices] = self.mask_id
        return x_noisy, mask_indices

    def compute_loss(self, logits: torch.Tensor, target: torch.Tensor, mask_indices: torch.Tensor) -> torch.Tensor:
        """
        CrossEntropy on masked positions only
        """
        vocab_size = logits.size(-1)
        loss_fct = nn.CrossEntropyLoss(reduction='none')
        logits_flat = logits.view(-1, vocab_size)
        target_flat = target.view(-1)
        losses = loss_fct(logits_flat, target_flat).view_as(target)
        masked = mask_indices.float()
        return (losses * masked).sum() / masked.sum().clamp(min=1)
    def train(self, dataloader, batch_size: int = 32, epochs: int = 10, save_path: str = '/kaggle/working/model'):  # <-- save_path added
        self.model.to(self.device)
        self.model.train()
        lr_scheduler = get_scheduler(
            name='cosine',
            optimizer=self.optimizer,
            num_warmup_steps=100,
            num_training_steps=len(dataloader) * epochs
        )

        epoch_losses = []  # <-- Track losses for plotting
        print('training started')
        for epoch in range(epochs):
            total_loss = 0.0
            c =0
            for batch in dataloader:
                rgb_img = torch.tensor(batch['obs']['image']).to(self.device)
                
                gripper_state = torch.tensor(torch.cat((batch['obs']['pose'], batch['obs']['gripper']), dim=-1)).to(self.device)
                target_states = batch['target_state']

                image_features = self.model['vision_encoder'](rgb_img)
                obs_features = torch.cat([image_features, gripper_state], dim=-1)
                input_ids = self.tokenizer(target_states, padding=True, max_length=150)
                input_ids = torch.tensor(input_ids, device=self.device)

                t = torch.randint(0, self.T, input_ids.shape, device=self.device)
                x_noisy, mask_indices = self.q_sample(input_ids, t)
                # x_noisy= x_noisy.to(self.device)
                # obs_features= obs_features.to(self.device)
                out = self.model['lldm'](x_noisy, obs_features)
                logits = out

                loss = self.compute_loss(logits, input_ids, mask_indices)
                if c%10==0:
                   print(loss.item())
                if c==len(dataloader)/2:
                    print('saved')
                    torch.save(self.model.state_dict(), f"{save_path}_epoch{epoch+1}.pt")
                c+=1
                loss.backward()
                self.optimizer.step()
                self.optimizer.zero_grad()
                lr_scheduler.step()
                total_loss += loss.item()

            avg_loss = total_loss / len(dataloader)
            epoch_losses.append(avg_loss)
            print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}")

            # <-- Save model after each epoch
            torch.save(self.model.state_dict(), f"{save_path}_epoch{epoch+1}.pt")

        # <-- Plot loss after training
        plt.plot(range(1, epochs + 1), epoch_losses, marker='o')
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title("Training Loss Over Epochs")
        plt.grid()
        plt.savefig("training_loss.png")
        plt.show()


In [7]:

# -----------------------------------------------------------------------------
# Inference sampler: reverse diffusion via Gumbel-max & confidence-based unmasking
# -----------------------------------------------------------------------------

def add_gumbel_noise(logits: torch.Tensor, temperature: float):
    if temperature == 0:
        return logits
    logits = logits.to(torch.float64)
    noise = torch.rand_like(logits, dtype=torch.float64)
    gumbel = (-torch.log(noise)) ** temperature
    return logits.exp() / gumbel


def get_num_transfer_tokens(mask_index: torch.Tensor, steps: int):
    mask_num = mask_index.sum(dim=1, keepdim=True)
    base = mask_num // steps
    rem = mask_num % steps
    schedule = base.repeat(1, steps)
    for i in range(mask_num.size(0)):
        schedule[i, :rem[i]] += 1
    return schedule.long()

@torch.no_grad()
def generate(model, prompt: torch.Tensor,
             steps: int = 128, gen_length: int = 128,
             block_length: int = 32,
             temperature: float = 0.0,
             cfg_scale: float = 0.0,
             remasking: str = 'low_confidence',
             mask_id: int = None):
    device = model.device
    mask_id = mask_id or model.config.mask_token_id
    B = gen_length // block_length
    steps_per_block = steps // B

    # init: keep prompt, mask the generation region
    x = torch.full((1, prompt.size(1)+gen_length), mask_id, dtype=torch.long, device=device)
    x[0, :prompt.size(1)] = prompt
    prompt_mask = x != mask_id

    for b in range(B):
        start = prompt.size(1) + b*block_length
        end = start + block_length
        mask_idx = x[:, start:end] == mask_id
        num_tokens = get_num_transfer_tokens(mask_idx, steps_per_block)
        for t in range(steps_per_block):
            mask_all = x == mask_id
            # classifier-free guidance
            if cfg_scale > 0:
                uncond = x.clone()
                uncond[prompt_mask] = mask_id
                cat = torch.cat([x, uncond], dim=0)
                logits = model(cat).logits
                cond, uncond_logits = logits.chunk(2, dim=0)
                logits = uncond_logits + (cfg_scale+1)*(cond - uncond_logits)
            else:
                logits = model(x).logits

            logits_noise = add_gumbel_noise(logits, temperature)
            x0 = logits_noise.argmax(dim=-1)

            if remasking == 'low_confidence':
                probs = F.softmax(logits.to(torch.float64), dim=-1)
                confid = probs.gather(-1, x0.unsqueeze(-1)).squeeze(-1)
            else:
                confid = torch.rand_like(x, dtype=torch.float64)
            confid[:, :start+block_length] = -np.inf

            # pick top-k_t
            k_t = num_tokens[:, t]
            transfer = torch.zeros_like(x, dtype=torch.bool)
            for i in range(x.size(0)):
                _, idx = torch.topk(confid[i], k=k_t[i].item())
                transfer[i, idx] = True

            x[transfer] = x0[transfer]

    return x
# -----------------------------------------------------------------------------
# Main entry points
# -----------------------------------------------------------------------------
from torchvision import transforms


mode = 'train'
batch_size = 512
epochs = 10
tokeniser_path = "/kaggle/input/d/dhruvsinghsachan/pickan/" 
if mode == 'train':
    print('loading_dataset')
    ds = ManiSkillSequenceDataset('/kaggle/input/grippp/out_dataset', transform=transforms.ToTensor())
    print('loaded')
    loader = DataLoader(ds, batch_size=8, shuffle=True, num_workers=4)
    tokeniser = TemporalBPEProcessor.load(tokeniser_path)
    
# else:
#     device = 'cuda' if torch.cuda.is_available() else 'cpu'
#     device = torch.device(device)
    
#         # out = generate(model, input_ids, steps=128, gen_length=128, block_length=32,
#         #                temperature=0.0, cfg_scale=0.0)
#         # print(tokenizer.batch_decode(out[:, input_ids.size(1):], skip_special_tokens=True)[0])


loading_dataset
loaded


In [ ]:
# import torch, gc
# torch.cuda.empty_cache()
# gc.collect()
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
trainer = DiffusionTrainer2(tokeniser=tokeniser, vocab_size=500, device= 'cuda' if torch.cuda.is_available() else 'cpu', load_path = '')
trainer.train(loader, batch_size=batch_size, epochs=epochs)


training started
